# F1 Strategy: GRU Multi-Task Model + Strategy Simulator

This notebook combines:
- GRU + MLP multi-task model (stops cls, tire cls, time reg)
- Strategy enumeration/simulation (0..3 stops)
- Ranked output with total predicted race time and stint breakdown
- Train/validation plots for loss and accuracy

In [1]:
import re
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
DATA_DIR = Path("openf1_data")
assert DATA_DIR.exists(), f"Missing: {DATA_DIR.resolve()}"

MODEL_FILE = "multitask_strategy_model.keras"
PREPROC_FILE = "multitask_preprocessing.joblib"

RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather.csv"

WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]
DEFAULT_PIT_LOSS = 22.0

In [3]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None

def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def safe_mode(series, default="MEDIUM"):
    s = pd.Series(series).dropna()
    if s.empty:
        return default
    m = s.mode()
    return m.iloc[0] if not m.empty else default

def format_time(sec):
    m = int(sec // 60)
    s = sec - m * 60
    return f"{m}:{s:06.3f}"

In [4]:
def load_session_maps():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}, {}

    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    sessions["track_name"] = sessions["circuit_short_name"].fillna(sessions["location"]).fillna(sessions["meeting_key"].astype(str))
    return (
        dict(zip(sessions["session_key"], sessions["track_name"])),
        dict(zip(sessions["session_key"], sessions["meeting_key"])),
        dict(zip(sessions["session_key"], sessions["session_name"].astype(str))),
    )

session_to_track, session_to_meeting, session_to_name = load_session_maps()

## Build training data for GRU multi-task

In [5]:
def get_weather_seq_and_summary(session_key):
    w = safe_read_csv(DATA_DIR / f"weather_session_{session_key}.csv")
    if w.empty:
        return None, None

    for c in WEATHER_FEATURES:
        if c not in w.columns:
            w[c] = np.nan
        w[c] = pd.to_numeric(w[c], errors="coerce")

    w = w.dropna(subset=WEATHER_FEATURES, how="all")
    if w.empty:
        return None, None

    seq = w[WEATHER_FEATURES].ffill().bfill().fillna(0.0).values
    summary = {
        "air_temp_mean": float(w["air_temperature"].mean()),
        "track_temp_mean": float(w["track_temperature"].mean()),
        "humidity_mean": float(w["humidity"].mean()),
        "rain_minutes_ratio": float((w["rainfall"].fillna(0) > 0).mean()),
        "wind_speed_mean": float(w["wind_speed"].mean()),
    }
    return seq, summary

def estimate_time_loss_target(session_key, team_name, driver_team):
    p = safe_read_csv(DATA_DIR / f"pit_session_{session_key}.csv")
    if p.empty or "driver_number" not in p.columns:
        return 0.0
    p = p.merge(driver_team, on="driver_number", how="left")
    p = p[p["team_name"] == team_name]
    if p.empty:
        return 0.0
    # If no duration, nominal
    for c in ["pit_duration", "duration", "pit_time"]:
        if c in p.columns:
            d = pd.to_numeric(p[c], errors="coerce").dropna()
            if len(d):
                return float(d.sum() + 15.0 * len(d))
    return float(21.0 * len(p))

def build_samples(race_only=True):
    rows, seqs = [], []
    for sf in sorted(DATA_DIR.glob("stints_session_*.csv")):
        sid = parse_session_key(sf)
        if sid is None:
            continue
        sname = session_to_name.get(sid, "")
        if race_only and ("Race" not in sname and "RACE" not in sname):
            continue

        st = safe_read_csv(sf)
        dr = safe_read_csv(DATA_DIR / f"drivers_session_{sid}.csv")
        if st.empty or dr.empty:
            continue

        for c in ["driver_number", "compound", "stint_number"]:
            if c not in st.columns:
                st[c] = np.nan
        for c in ["driver_number", "team_name"]:
            if c not in dr.columns:
                dr[c] = np.nan

        driver_team = dr[["driver_number", "team_name"]].dropna().drop_duplicates()
        st = st.merge(driver_team, on="driver_number", how="left")

        pits = safe_read_csv(DATA_DIR / f"pit_session_{sid}.csv")
        if pits.empty:
            pits = pd.DataFrame(columns=["driver_number"])
        if "driver_number" not in pits.columns:
            pits["driver_number"] = np.nan
        pits = pits.merge(driver_team, on="driver_number", how="left")

        grid = safe_read_csv(DATA_DIR / f"starting_grid_session_{sid}.csv")
        pos_col = "position" if "position" in grid.columns else ("grid_position" if "grid_position" in grid.columns else None)
        if not grid.empty and pos_col and "driver_number" in grid.columns:
            g = grid[["driver_number", pos_col]].copy()
            g.columns = ["driver_number", "starting_position"]
            g["starting_position"] = pd.to_numeric(g["starting_position"], errors="coerce")
            g = g.merge(driver_team, on="driver_number", how="left")
            team_pos = g.groupby("team_name")["starting_position"].mean().to_dict()
        else:
            team_pos = {}

        seq, ws = get_weather_seq_and_summary(sid)
        if seq is None:
            continue

        track = str(session_to_track.get(sid, session_to_meeting.get(sid, "UNKNOWN_TRACK")))

        for team, t_st in st.groupby("team_name", dropna=True):
            team = str(team)
            t_pits = pits[pits["team_name"] == team]
            pit_stops = int(len(t_pits))

            s1 = t_st[t_st["stint_number"] == 1]
            start_comp = safe_mode(s1["compound"] if len(s1) else t_st["compound"], default="MEDIUM")
            start_comp = str(start_comp).upper()
            if start_comp not in VALID_COMPOUNDS:
                start_comp = "MEDIUM"

            non_start = t_st[t_st["compound"].astype(str).str.upper() != start_comp]["compound"]
            target_comp = str(safe_mode(non_start, default=safe_mode(t_st["compound"], "MEDIUM"))).upper()
            if target_comp not in VALID_COMPOUNDS:
                target_comp = "MEDIUM"

            rows.append({
                "session_key": sid,
                "team_name": team,
                "track_name": track,
                "starting_position": float(team_pos.get(team, 10.0)),
                "starting_compound": start_comp,
                "pit_stops": pit_stops,
                "target_compound": target_comp,
                "time_loss_target": estimate_time_loss_target(sid, team, driver_team),
                **ws,
            })
            seqs.append(seq)
    return pd.DataFrame(rows), seqs

df, X_seq_raw = build_samples(race_only=True)
print(len(df))
df.head()

90


,session_key,team_name,track_name,starting_position,starting_compound,pit_stops,target_compound,time_loss_target,air_temp_mean,track_temp_mean,humidity_mean,rain_minutes_ratio,wind_speed_mean
0,10033,Alpine,Miami,10.0,HARD,1,MEDIUM,37.060,26.573154,38.690604,63.020134,0.033557,1.04698
1,10033,Aston Martin,Miami,10.0,HARD,2,MEDIUM,75.919,26.573154,38.690604,63.020134,0.033557,1.04698
2,10033,Ferrari,Miami,10.0,HARD,2,MEDIUM,75.287,26.573154,38.690604,63.020134,0.033557,1.04698
3,10033,Haas F1 Team,Miami,10.0,HARD,1,MEDIUM,38.162,26.573154,38.690604,63.020134,0.033557,1.04698
4,10033,Kick Sauber,Miami,10.0,HARD,2,MEDIUM,74.296,26.573154,38.690604,63.020134,0.033557,1.04698


In [6]:
team_le = LabelEncoder(); track_le = LabelEncoder(); start_le = LabelEncoder()
stops_le = LabelEncoder(); tire_le = LabelEncoder()

df["team_id"] = team_le.fit_transform(df["team_name"].astype(str))
df["track_id"] = track_le.fit_transform(df["track_name"].astype(str))
df["start_id"] = start_le.fit_transform(df["starting_compound"].astype(str))

y_stops = stops_le.fit_transform(df["pit_stops"])
y_tire = tire_le.fit_transform(df["target_compound"].astype(str))
y_time_raw = df["time_loss_target"].astype(float).values
y_time_mean = y_time_raw.mean()
y_time_std = y_time_raw.std() + 1e-8
y_time = ((y_time_raw - y_time_mean) / y_time_std).astype("float32")

NUM_COLS = ["starting_position", "air_temp_mean", "track_temp_mean", "humidity_mean", "rain_minutes_ratio", "wind_speed_mean"]
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df[c] = df[c].fillna(df[c].median())

X_num = df[NUM_COLS].values.astype("float32")
X_team = df["team_id"].values.astype("int32")
X_track = df["track_id"].values.astype("int32")
X_start = df["start_id"].values.astype("int32")
X_seq = pad_sequences(X_seq_raw, padding="post", dtype="float32")

idx = np.arange(len(df))
tr_idx, va_idx = train_test_split(idx, test_size=0.2, random_state=42)

X_train = {
    "weather_seq": X_seq[tr_idx],
    "team_in": X_team[tr_idx],
    "track_in": X_track[tr_idx],
    "start_comp_in": X_start[tr_idx],
    "num_in": X_num[tr_idx],
}
X_val = {
    "weather_seq": X_seq[va_idx],
    "team_in": X_team[va_idx],
    "track_in": X_track[va_idx],
    "start_comp_in": X_start[va_idx],
    "num_in": X_num[va_idx],
}
y_train = {"stops_head": y_stops[tr_idx], "tire_head": y_tire[tr_idx], "time_head": y_time[tr_idx]}
y_val = {"stops_head": y_stops[va_idx], "tire_head": y_tire[va_idx], "time_head": y_time[va_idx]}

## Build and train GRU multi-task model

In [7]:
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

# Optional: reproducibility
keras.utils.set_random_seed(42)

T, F = X_seq.shape[1], X_seq.shape[2]
n_team, n_track, n_start = len(team_le.classes_), len(track_le.classes_), len(start_le.classes_)
n_stops, n_tire = len(stops_le.classes_), len(tire_le.classes_)

# Inputs
weather_in = keras.Input(shape=(T, F), name="weather_seq")
team_in = keras.Input(shape=(1,), dtype="int32", name="team_in")
track_in = keras.Input(shape=(1,), dtype="int32", name="track_in")
start_in = keras.Input(shape=(1,), dtype="int32", name="start_comp_in")
num_in = keras.Input(shape=(len(NUM_COLS),), dtype="float32", name="num_in")

# Sequence encoder (stronger than single GRU)
w = layers.Masking(mask_value=0.0)(weather_in)
w = layers.Bidirectional(
    layers.GRU(64, return_sequences=True, dropout=0.15, recurrent_dropout=0.10)
)(w)
w = layers.Bidirectional(
    layers.GRU(48, return_sequences=False, dropout=0.15, recurrent_dropout=0.10)
)(w)
w = layers.LayerNormalization()(w)
w = layers.Dropout(0.20)(w)

# Categorical embeddings (slightly larger, but bounded)
team_emb_dim = min(24, max(6, n_team // 2))
track_emb_dim = min(24, max(6, n_track // 2))
start_emb_dim = min(10, max(4, n_start // 2))

te = layers.Embedding(n_team, team_emb_dim, name="team_emb")(team_in)
te = layers.Flatten()(te)

tr = layers.Embedding(n_track, track_emb_dim, name="track_emb")(track_in)
tr = layers.Flatten()(tr)

sc = layers.Embedding(n_start, start_emb_dim, name="start_emb")(start_in)
sc = layers.Flatten()(sc)

# Numeric branch
n = layers.LayerNormalization()(num_in)
n = layers.Dense(32, activation="relu")(n)
n = layers.Dropout(0.10)(n)

# Shared trunk
x = layers.Concatenate()([w, te, tr, sc, n])
x = layers.Dense(192, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.LayerNormalization()(x)
x = layers.Dropout(0.30)(x)

x = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
x = layers.LayerNormalization()(x)
x = layers.Dropout(0.20)(x)

# Head-specific towers
st = layers.Dense(64, activation="relu")(x)
st = layers.Dropout(0.20)(st)
stops_out = layers.Dense(n_stops, activation="softmax", name="stops_head")(st)

ti = layers.Dense(64, activation="relu")(x)
ti = layers.Dropout(0.20)(ti)
tire_out = layers.Dense(n_tire, activation="softmax", name="tire_head")(ti)

tm = layers.Dense(64, activation="relu")(x)
tm = layers.Dropout(0.15)(tm)
time_out = layers.Dense(1, activation="linear", name="time_head")(tm)

model = keras.Model(
    [weather_in, team_in, track_in, start_in, num_in],
    [stops_out, tire_out, time_out]
)

# Optimizer: AdamW if available, else Adam
try:
    opt = keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4)
except Exception:
    opt = keras.optimizers.Adam(learning_rate=3e-4)

model.compile(
    optimizer=opt,
    loss={
        "stops_head": "sparse_categorical_crossentropy",
        "tire_head": "sparse_categorical_crossentropy",
        "time_head": "huber"   # more robust than MSE on small/noisy data
    },
    loss_weights={
        "stops_head": 1.3,
        "tire_head": 1.2,
        "time_head": 0.7
    },
    metrics={
        "stops_head": ["accuracy"],
        "tire_head": ["accuracy"],
        "time_head": ["mae"]
    },
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True, min_delta=1e-4
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=16,   # small data => smaller batch can help generalization
    callbacks=callbacks,
    verbose=1,
)

2026-04-27 13:08:48.100163: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-04-27 13:08:48.100207: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-04-27 13:08:48.100213: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-04-27 13:08:48.100229: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-27 13:08:48.100244: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/150


2026-04-27 13:08:50.838810: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


2/5 ━━━━━━━━━━━━━━━━━━━━ 13:49 276s/step - loss: 8.2458 - stops_head_accuracy: 0.0938 - stops_head_loss: 3.4308 - time_head_loss: 0.7860 - time_head_mae: 1.1948 - tire_head_accuracy: 0.2188 - tire_head_loss: 2.6698

KeyboardInterrupt: 

In [ ]:
h = history.history
ep = range(1, len(h["loss"]) + 1)

fig, ax = plt.subplots(2, 2, figsize=(13, 9))
ax[0,0].plot(ep, h["loss"], label="train"); ax[0,0].plot(ep, h["val_loss"], label="val"); ax[0,0].set_title("Total Loss"); ax[0,0].legend(); ax[0,0].grid(alpha=.3)
ax[0,1].plot(ep, h["stops_head_accuracy"], label="train"); ax[0,1].plot(ep, h["val_stops_head_accuracy"], label="val"); ax[0,1].set_title("Stops Accuracy"); ax[0,1].legend(); ax[0,1].grid(alpha=.3)
ax[1,0].plot(ep, h["tire_head_accuracy"], label="train"); ax[1,0].plot(ep, h["val_tire_head_accuracy"], label="val"); ax[1,0].set_title("Tire Accuracy"); ax[1,0].legend(); ax[1,0].grid(alpha=.3)
ax[1,1].plot(ep, h["time_head_loss"], label="train"); ax[1,1].plot(ep, h["val_time_head_loss"], label="val"); ax[1,1].set_title("Time Head Loss (MSE)"); ax[1,1].legend(); ax[1,1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Save model + preprocessing

In [ ]:
model.save(MODEL_FILE)
joblib.dump({
    "team_le": team_le,
    "track_le": track_le,
    "start_le": start_le,
    "stops_le": stops_le,
    "tire_le": tire_le,
    "num_cols": NUM_COLS,
    "weather_features": WEATHER_FEATURES,
    "max_timesteps": X_seq.shape[1],
    "y_time_mean": float(y_time_mean),
    "y_time_std": float(y_time_std),
}, PREPROC_FILE)
print("Saved", MODEL_FILE, PREPROC_FILE)

## Strategy simulation using the GRU model (top strategies + consistent stint breakdown)

In [ ]:
def encode_with_unknown(le, val):
    val = str(val)
    if val in set(le.classes_):
        return int(le.transform([val])[0])
    return 0

def estimate_track_pit_loss_map():
    out = {}
    for pf in sorted(DATA_DIR.glob("pit_session_*.csv")):
        sid = parse_session_key(pf)
        if sid is None:
            continue
        track = str(session_to_track.get(sid, "UNKNOWN_TRACK"))
        p = safe_read_csv(pf)
        if p.empty:
            continue
        # no reliable universal duration col; fallback to nominal from count if missing
        dcol = next((c for c in ["pit_duration", "duration", "pit_time"] if c in p.columns), None)
        if dcol:
            d = pd.to_numeric(p[dcol], errors="coerce").dropna()
            if len(d):
                out[track] = float(np.median(d) + 15.0)
    return out

def forecast_to_sequence_and_summary(forecast_df, weather_features, max_timesteps):
    f = forecast_df.copy()
    for c in weather_features:
        if c not in f.columns:
            f[c] = np.nan
        f[c] = pd.to_numeric(f[c], errors="coerce")
    f = f[weather_features].ffill().bfill().fillna(0.0)
    seq = pad_sequences([f.values.astype("float32")], maxlen=max_timesteps, padding="post", dtype="float32")
    summary = {
        "air_temp_mean": float(f["air_temperature"].mean()),
        "track_temp_mean": float(f["track_temperature"].mean()),
        "humidity_mean": float(f["humidity"].mean()),
        "rain_minutes_ratio": float((f["rainfall"] > 0).mean()),
        "wind_speed_mean": float(f["wind_speed"].mean()),
    }
    return seq, summary

def split_laps(total_laps, n_stints):
    b = total_laps // n_stints
    r = total_laps % n_stints
    return [b + (1 if i < r else 0) for i in range(n_stints)]

def build_candidates(start_comp, max_stops=3, wet=False):
    pool = VALID_COMPOUNDS if wet else DRY_COMPOUNDS
    start_comp = str(start_comp).upper()
    if start_comp not in pool:
        start_comp = "MEDIUM" if "MEDIUM" in pool else pool[0]

    cands = []
    for s in range(max_stops + 1):
        if s == 0:
            cands.append([start_comp])
        else:
            for seq in product(pool, repeat=s):
                full = [start_comp] + list(seq)
                if len(set(full)) == 1:  # avoid pointless same compound every stint
                    continue
                cands.append(full)
    return cands

def simulate_with_gru(model, pre, race_context, seq_input, weather_summary, compounds, pit_loss):
    team_id = encode_with_unknown(pre["team_le"], race_context["team_name"])
    track_id = encode_with_unknown(pre["track_le"], race_context["track_name"])
    start_id = encode_with_unknown(pre["start_le"], race_context["starting_compound"])

    num_vec = np.array([[race_context["starting_position"], weather_summary["air_temp_mean"], weather_summary["track_temp_mean"], weather_summary["humidity_mean"], weather_summary["rain_minutes_ratio"], weather_summary["wind_speed_mean"]]], dtype="float32")

    p_stops, p_tire, p_time = model.predict({
        "weather_seq": seq_input,
        "team_in": np.array([team_id], dtype=np.int32),
        "track_in": np.array([track_id], dtype=np.int32),
        "start_comp_in": np.array([start_id], dtype=np.int32),
        "num_in": num_vec,
    }, verbose=0)

    # Base total time estimate from historical pace proxy:
    # approximate avg lap by track temp/rain (simple heuristic baseline)
    total_laps = int(race_context["race_laps"])
    base_lap = 92.0 + 0.08 * max(weather_summary["track_temp_mean"] - 35.0, 0) + 6.0 * weather_summary["rain_minutes_ratio"]
    base_total = base_lap * total_laps

    # Compound delta heuristic per lap (relative)
    comp_delta = {"SOFT": -0.25, "MEDIUM": 0.0, "HARD": 0.18, "INTERMEDIATE": 0.9, "WET": 1.8}
    # Degradation per lap in stint
    degr = {"SOFT": 0.035, "MEDIUM": 0.024, "HARD": 0.018, "INTERMEDIATE": 0.030, "WET": 0.040}

    stint_laps = split_laps(total_laps, len(compounds))
    detailed = []
    sim_time = 0.0
    for i, (c, L) in enumerate(zip(compounds, stint_laps), start=1):
        c = str(c).upper()
        lap_times = []
        for a in range(L):
            lap_t = base_lap + comp_delta.get(c, 0.0) + degr.get(c, 0.025) * a
            lap_times.append(lap_t)
        st_sec = float(np.sum(lap_times))
        sim_time += st_sec
        detailed.append({"stint": i, "compound": c, "laps": L, "predicted_stint_time_sec": st_sec})
        if i < len(compounds):
            sim_time += pit_loss

    # Blend with GRU time-head prediction as calibration term
    gru_time_norm = float(p_time[0][0])
    gru_time_loss = gru_time_norm * pre["y_time_std"] + pre["y_time_mean"]
    pred_stops_class = int(np.argmax(p_stops[0]))
    pred_stops_val = int(pre["stops_le"].inverse_transform([pred_stops_class])[0])

    # penalty if candidate stops diverges a lot from model preference
    stop_mismatch_penalty = abs((len(compounds)-1) - pred_stops_val) * 4.0

    total = sim_time + 0.35 * gru_time_loss + stop_mismatch_penalty
    return {
        "strategy": " -> ".join(compounds),
        "stops": len(compounds)-1,
        "predicted_total_time_sec": total,
        "pit_loss_per_stop_sec": pit_loss,
        "gru_predicted_stops": pred_stops_val,
        "gru_tire_class": pre["tire_le"].inverse_transform([int(np.argmax(p_tire[0]))])[0],
        "gru_time_loss_sec": gru_time_loss,
        "stints": detailed,
    }

In [ ]:
# Example inference files if missing
if not Path(RACE_CONTEXT_FILE).exists():
    pd.DataFrame([{
        "team_name": "Ferrari",
        "track_name": "Imola",
        "starting_position": 4,
        "starting_compound": "MEDIUM",
        "race_laps": 63
    }]).to_csv(RACE_CONTEXT_FILE, index=False)

if not Path(FORECAST_FILE).exists():
    pd.DataFrame([
        {"date":"2026-05-17T13:00:00Z","air_temperature":22.1,"track_temperature":37.5,"humidity":44,"rainfall":0.0,"wind_speed":2.9},
        {"date":"2026-05-17T13:10:00Z","air_temperature":22.4,"track_temperature":38.1,"humidity":43,"rainfall":0.0,"wind_speed":3.1},
        {"date":"2026-05-17T13:20:00Z","air_temperature":22.8,"track_temperature":38.9,"humidity":41,"rainfall":0.0,"wind_speed":2.7},
    ]).to_csv(FORECAST_FILE, index=False)

pre = joblib.load(PREPROC_FILE)
gru_model = keras.models.load_model(MODEL_FILE)
pit_loss_map = estimate_track_pit_loss_map()

rc = pd.read_csv(RACE_CONTEXT_FILE).iloc[0].to_dict()
fc = pd.read_csv(FORECAST_FILE)

rc["team_name"] = str(rc.get("team_name", "UNKNOWN_TEAM"))
rc["track_name"] = str(rc.get("track_name", "UNKNOWN_TRACK"))
rc["starting_position"] = float(rc.get("starting_position", 10))
rc["starting_compound"] = str(rc.get("starting_compound", "MEDIUM")).upper()
rc["race_laps"] = int(rc.get("race_laps", 60))

seq_input, ws = forecast_to_sequence_and_summary(fc, pre["weather_features"], pre["max_timesteps"])
wet = ws["rain_minutes_ratio"] > 0.2
cands = build_candidates(rc["starting_compound"], max_stops=3, wet=wet)
pit_loss = float(pit_loss_map.get(rc["track_name"], DEFAULT_PIT_LOSS))

results = [simulate_with_gru(gru_model, pre, rc, seq_input, ws, c, pit_loss) for c in cands]
res_df = pd.DataFrame([{k:v for k,v in r.items() if k != "stints"} for r in results]).sort_values("predicted_total_time_sec").reset_index(drop=True)
best = float(res_df.loc[0, "predicted_total_time_sec"])
res_df["delta_to_best_sec"] = res_df["predicted_total_time_sec"] - best

show = res_df.head(10).copy()
show["predicted_total_time"] = show["predicted_total_time_sec"].apply(format_time)
show["delta_to_best"] = show["delta_to_best_sec"].map(lambda x: f"+{x:.3f}s")

print("\n--- TOP 10 STRATEGIES ---\n")
display(show[["strategy", "stops", "predicted_total_time", "delta_to_best", "pit_loss_per_stop_sec"]])

best_strategy_name = res_df.loc[0, "strategy"]
best_obj = next(r for r in results if r["strategy"] == best_strategy_name)

print("=== RECOMMENDED STRATEGY ===")
print("Team:", rc["team_name"])
print("Track:", rc["track_name"])
print("Start pos:", rc["starting_position"])
print("Starting tire:", rc["starting_compound"])
print("Recommended sequence:", best_obj["strategy"])
print("Stops:", best_obj["stops"])
print("Predicted total time:", format_time(best_obj["predicted_total_time_sec"]))
print("Assumed pit loss per stop:", f"{best_obj['pit_loss_per_stop_sec']:.2f}s")

print("\nStint breakdown (best):")
for st in best_obj["stints"]:
    print(f"  Stint {st['stint']}: {st['compound']} | laps={st['laps']} | predicted={format_time(st['predicted_stint_time_sec'])}")